# Rail Corrugation: from data to model selection

This is the main notebook for the Rail Corrugation project. Run its cells from top to bottom using **Restart Kernel and Run All**.

Everything needed for this stage is here: EDA, feature calculations, baseline training, error review, refinement, and model selection. It uses ordinary Python libraries and the labelled `Dataset/Train` files; it does not import our helper files or require previous notebook outputs or caches.

We keep six key experiments from the earlier investigation. The feature calculations, model settings, file order, and validation splits are preserved so those results can be reproduced. Earlier notebooks retain the full experiment history.

**Our question:** can train vibration distinguish Normal, Side I, and Side II, even when high speed makes healthy trains vibrate more?

1. Inspect the data and class balance.
2. Turn each recording into a small table of measurements.
3. Check speed and side-specific patterns.
4. Train and compare simple models.
5. Review errors and reproduce the strongest refinement.
6. Select the model for the next deployment stage.

A complete run reads about 4.38 GB, one CSV at a time, and trains temporary validation models. Allow a few minutes. No cache is required: each run recomputes its evidence.


## 1. Setup

Choose a Python kernel with NumPy, pandas, SciPy, matplotlib, seaborn, and scikit-learn installed. The earlier run used NumPy 2.2.6, pandas 2.2.3, SciPy 1.17.0, and scikit-learn 1.8.0. The cell prints your versions; library changes can alter exact results.

The path lookup works from the repository root, this folder, or its `notebooks` folder.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy import signal, stats
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, confusion_matrix

try:
    from IPython.display import display
except ImportError:
    display = print  # Allows the same cells to run in a plain Python check.

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 12)
SEED = 42
CLASSES = ["Normal", "Side I", "Side II"]
FS = 10_000
EPSILON = 1e-12
BANDS = [(0, 50), (50, 100), (100, 250), (250, 1_000), (1_000, 5_000)]

cwd = Path.cwd().resolve()
DATASET_DIR = None
for root in [cwd, *cwd.parents]:
    candidates = [
        root / "Dataset",
        root / "Backend" / "Rail_Corrugation" / "Dataset",
        root / "Backend" / "02_Datasets" / "Rail_Corrugation",
    ]
    for candidate in candidates:
        if (candidate / "Train").is_dir() and (candidate / "Train_Labels.csv").is_file():
            DATASET_DIR = candidate
            break
    if DATASET_DIR is not None:
        break
if DATASET_DIR is None:
    raise FileNotFoundError("Place Train/ and Train_Labels.csv inside Rail_Corrugation/Dataset.")

TRAIN_DIR = DATASET_DIR / "Train"
print("Dataset:", DATASET_DIR)
print("Versions:", {"numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "sklearn": sklearn.__version__})


## 2. Understand the files and class imbalance

One CSV is one example: 10,000 time samples across 129 columns, representing one second.

| Input | Meaning |
|---|---|
| First column | Wheel rotation pulse signal, used to estimate speed |
| Remaining 128 columns | Vibration and shock from 8 cars × 8 axle positions |
| Positions 1, 3, 5, 7 | Side I |
| Positions 2, 4, 6, 8 | Side II |

The three labels describe the **whole recording**. Normal means both rails are normal; Side I or Side II indicates the affected rail.

First check that every label has exactly one file, then plot the class counts. Keep the label-file order: changing row order changes seeded validation folds.


In [ ]:
labels = pd.read_csv(DATASET_DIR / "Train_Labels.csv")
if not {"filename", "label"}.issubset(labels.columns):
    raise ValueError("Train_Labels.csv needs filename and label columns.")
labels = labels[["filename", "label"]].copy()
if labels.isna().any().any() or labels["filename"].duplicated().any():
    raise ValueError("Missing labels or duplicate filenames.")
if set(labels["label"]) != set(CLASSES):
    raise ValueError("Expected exactly Normal, Side I, and Side II.")
if set(labels["filename"]) != {p.name for p in TRAIN_DIR.glob("*.csv")}:
    raise ValueError("Training filenames and labels do not match.")

counts = labels["label"].value_counts().reindex(CLASSES)
display(counts.rename("Files").to_frame())
ax = counts.plot.bar(color=["#4C78A8", "#F58518", "#E45756"], figsize=(6, 3))
ax.bar_label(ax.containers[0])
ax.set(xlabel="", ylabel="Files", title="Many Normal files, few fault examples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Make one small summary per file

A feature is just a measurement: for example, the largest vibration RMS on Side I. RMS measures signal magnitude over time.

The function below computes the same 69 measurements as the earlier feature module:
- Two speed measurements.
- Vibration and shock summaries for each rail side: RMS, peaks, kurtosis (spikiness), and spectral power.
- Ratios comparing the two sides within the same recording.

Only 15 vibration-time measurements are used by the current leading model. The other measurements let us reproduce the comparisons that led to that choice.

These functions are defined **inside this notebook**. They use no labels and learn nothing across files. Scaling and classifier training happen later, inside validation folds. Open the collapsed calculation cell if you want to inspect the recipe.


In [ ]:
# Header order and side assignments come from the Info Kit.
EXPECTED_HEADER = ["Rotating speed"] + [
    f"{kind} of bearing in position {position} of car {car}"
    for car in range(1, 9) for position in range(1, 9)
    for kind in ["Vibration", "Shock"]
]
GROUP_COLUMNS = {
    (kind.lower(), side): [
        f"{kind} of bearing in position {position} of car {car}"
        for car in range(1, 9) for position in positions
    ]
    for kind in ["Vibration", "Shock"]
    for side, positions in [("side_i", [1, 3, 5, 7]), ("side_ii", [2, 4, 6, 8])]
}

def read_recording(path):
    frame = pd.read_csv(path, dtype=np.float32)
    if frame.shape != (10_000, 129) or frame.columns.tolist() != EXPECTED_HEADER:
        raise ValueError(f"{Path(path).name}: expected 10,000 rows and the 129 ordered sensor columns.")
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"{Path(path).name}: missing or non-finite measurements.")
    return frame

def finite_number(value):
    # Retain the original recipe's neutral value for undefined summaries,
    # e.g. kurtosis of a constant signal. Raw invalid inputs are rejected above.
    return float(value) if np.isfinite(value) else 0.0

def group_summary(values):
    rms = np.sqrt(np.mean(np.square(values), axis=0, dtype=np.float64))
    peak = np.max(np.abs(values), axis=0)
    kurtosis = stats.kurtosis(values, axis=0, fisher=True, bias=False)
    crest = peak / (rms + EPSILON)
    result = {
        "rms_median": finite_number(np.median(rms)),
        "rms_p90": finite_number(np.quantile(rms, 0.90)),
        "rms_max": finite_number(np.max(rms)),
        "abs_peak_max": finite_number(np.max(peak)),
        "kurtosis_median": finite_number(np.median(kurtosis)),
        "crest_factor_p90": finite_number(np.quantile(crest, 0.90)),
    }

    frequencies, psd = signal.welch(
        values, fs=FS, nperseg=2048, axis=0, detrend="constant", scaling="density"
    )
    mean_psd = psd.mean(axis=1)
    positive = frequencies > 0
    power = mean_psd[positive]
    frequency = frequencies[positive]
    total_power = np.trapezoid(power, frequency)
    mass = power.sum()
    probabilities = power / (mass + EPSILON)
    result.update({
        "dominant_frequency_hz": finite_number(frequency[np.argmax(power)]),
        "spectral_centroid_hz": finite_number((frequency * power).sum() / mass if mass > 0 else 0.0),
        "spectral_entropy": finite_number(
            -np.sum(probabilities * np.log(probabilities + EPSILON)) / np.log(len(probabilities))
        ),
    })
    # Preserve the previous integration boundaries to reproduce previous models.
    # Relative powers are band descriptors; they need not sum to exactly one.
    for low, high in BANDS:
        mask = (frequencies >= low) & (frequencies < high)
        band_power = np.trapezoid(mean_psd[mask], frequencies[mask]) if mask.sum() >= 2 else 0.0
        result[f"relative_power_{low}_{high}_hz"] = finite_number(band_power / (total_power + EPSILON))
        if (low, high) == (50, 100):
            result["absolute_power_50_100_hz"] = finite_number(band_power)
    return result

def extract_features(path):
    frame = read_recording(path)
    pulses = frame.iloc[:, 0].to_numpy(dtype=np.float64)
    high = pulses > (pulses.min() + pulses.max()) / 2
    transitions = np.count_nonzero(np.diff(high.astype(np.int8)))
    # 90 teeth, two transitions per tooth, wheel diameter 0.85 m.
    revolutions_per_second = transitions / (2 * 90 * (len(pulses) / FS))
    features = {
        "file_id": Path(path).name,
        "speed__kmh": float(revolutions_per_second * (np.pi * 0.85) * 3.6),
        "speed__is_moving": float(transitions > 0),
    }
    for (kind, side), columns in GROUP_COLUMNS.items():
        values = frame[columns].to_numpy(dtype=np.float32, copy=False)
        features.update({f"{kind}__{side}__{name}": value
                         for name, value in group_summary(values).items()})

    comparisons = {
        "vibration": ["rms_p90", "rms_max", "abs_peak_max",
                      "relative_power_50_100_hz", "absolute_power_50_100_hz"],
        "shock": ["rms_max", "abs_peak_max"],
    }
    for kind, metrics in comparisons.items():
        for metric in metrics:
            left = features[f"{kind}__side_i__{metric}"]
            right = features[f"{kind}__side_ii__{metric}"]
            features[f"{kind}__side_log_ratio__{metric}"] = finite_number(
                np.log((left + EPSILON) / (right + EPSILON))
            )
    if not np.isfinite(list(features.values())[1:]).all():
        raise ValueError(f"{Path(path).name}: invalid computed features.")
    return features


In [ ]:
# Recompute from Train data so results never depend on an old cache.
rows = []
for index, filename in enumerate(labels["filename"], start=1):
    rows.append(extract_features(TRAIN_DIR / filename))
    if index == 1 or index % 50 == 0 or index == len(labels):
        print(f"Processed {index}/{len(labels)}", flush=True)

feature_table = pd.DataFrame(rows)
feature_columns = [c for c in feature_table if c != "file_id"]
data = labels.merge(feature_table, left_on="filename", right_on="file_id",
                    validate="one_to_one", sort=False)
assert data["filename"].tolist() == labels["filename"].tolist()
assert len(feature_columns) == 69
assert np.isfinite(data[feature_columns].to_numpy()).all()
display(pd.Series({
    "Files checked": len(data),
    "Samples per file": 10_000,
    "Columns per file": 129,
    "Files failing shape, header, or finite-value checks": 0,
    "Measurements per file": len(feature_columns),
}, name="Result").to_frame())
print("All labelled files passed the checks. No test inputs were used.")


## 4. EDA: what might distinguish the classes?

Two observations matter most:
1. Fault examples are recorded mainly at higher speeds. Speed and signal magnitude may be a shortcut for the model.
2. Comparing the two sides may help distinguish general vibration from a localised defect.

The plots below show the full distributions, including overlap. A zero log ratio means both sides are equal; positive means stronger Side I; negative means stronger Side II. A constant speed signal gives a zero estimate, but does not independently prove that the train was stationary.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
views = [
    ("speed__kmh", "Estimated speed by class", "km/h"),
    ("vibration__side_log_ratio__rms_max", "Maximum RMS: log(Side I / Side II)", "Log ratio"),
    ("vibration__side_log_ratio__absolute_power_50_100_hz",
     "50-100 Hz power: log(Side I / Side II)", "Log ratio"),
]
for ax, (column, title, unit) in zip(axes, views):
    sns.boxplot(data=data, x="label", y=column, order=CLASSES, showfliers=False, ax=ax)
    sns.stripplot(data=data, x="label", y=column, order=CLASSES,
                  color="black", alpha=0.35, size=3, ax=ax)
    ax.set(title=title, xlabel="", ylabel=unit)
plt.tight_layout()
plt.show()

evidence = [v[0] for v in views]
print("All files: class medians")
display(data.groupby("label")[evidence].median().reindex(CLASSES).round(3))
overlap = data[data["speed__kmh"].between(40, 50)]
print("40-50 km/h: compare classes at similar estimated speeds")
display(overlap.groupby("label").agg(
    files=("file_id", "size"),
    side_i_max_rms=("vibration__side_i__rms_max", "median"),
    side_ii_max_rms=("vibration__side_ii__rms_max", "median"),
    band_log_ratio=("vibration__side_log_ratio__absolute_power_50_100_hz", "median"),
).reindex(CLASSES).round(3))


### See the underlying signals

The first labelled file from each class is shown below, chosen by filename order rather than appearance. The left plot shows vibration strength over one second; the right shows where vibration power falls across frequencies. These examples illustrate the data, while the all-file comparisons above support broader observations.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
time = np.arange(10_000) / FS
for row, label in enumerate(CLASSES):
    filename = labels.loc[labels["label"] == label, "filename"].iloc[0]
    frame = read_recording(TRAIN_DIR / filename)
    for side, color in [("side_i", "#F58518"), ("side_ii", "#4C78A8")]:
        values = frame[GROUP_COLUMNS[("vibration", side)]].to_numpy(dtype=np.float32)
        trace = np.sqrt(np.mean(np.square(values), axis=1, dtype=np.float64))
        frequency, psd = signal.welch(values, fs=FS, nperseg=2048, axis=0,
                                      detrend="constant", scaling="density")
        axes[row, 0].plot(time, trace, color=color, alpha=0.7, label=side)
        axes[row, 1].semilogy(frequency[1:], psd.mean(axis=1)[1:], color=color, label=side)
    axes[row, 0].set(title=f"{label}: {filename}", ylabel="Across-sensor RMS (m/s²)")
    axes[row, 1].set(title=f"{label}: vibration spectrum", ylabel="PSD ((m/s²)²/Hz)")
    axes[row, 0].legend()
axes[-1, 0].set_xlabel("Time (seconds)")
axes[-1, 1].set_xlabel("Frequency (Hz)")
plt.tight_layout()
plt.show()


## 5. Set up a fair practice exam

We repeatedly train on about 80% of the labelled files and evaluate on the remaining 20%. Every experiment gets exactly the same splits.

- **Macro F1:** an equal-weight score for the three classes; always predicting Normal performs poorly.
- **Repeated score:** mean and standard deviation across 5 folds × 5 repeats. The standard deviation describes variation across folds, not a confidence interval.
- **Out-of-fold (OOF) predictions:** one prediction per file from a fixed five-fold split, used for the confusion matrix and per-class scores.
- **High-speed check:** score those OOF predictions only on files at or above 40 km/h. This is a subset check, not proof that the model generalises to unseen operating speeds.

There are only 2-3 Side I validation files in each fold, so one mistake can noticeably change a score. No run/session IDs are provided, and file-level validation cannot rule out related recordings across folds.

Scaling is fitted inside each training fold. Filenames and labels are never model inputs. EDA and model choice have used these training files, so the scores are development estimates; the organisers' held-out test remains the final check.


In [ ]:
y = data["label"]
vibration = [c for c in feature_columns if c.startswith("vibration__")]
vibration_time = [c for c in vibration if not any(w in c for w in ("frequency", "spectral", "power"))]
all_signals = [c for c in feature_columns if not c.startswith("speed__")]
assert len(vibration_time) == 15

# Materialise splits once: every model sees exactly the same training/validation files.
repeated_splits = list(RepeatedStratifiedKFold(
    n_splits=5, n_repeats=5, random_state=SEED
).split(data, y))
fixed_splits = list(StratifiedKFold(
    n_splits=5, shuffle=True, random_state=SEED
).split(data, y))

def logistic():
    return make_pipeline(StandardScaler(), LogisticRegression(
        class_weight="balanced", max_iter=5000, random_state=SEED
    ))

def trees(max_features="sqrt"):
    return ExtraTreesClassifier(
        n_estimators=250, min_samples_leaf=2, max_features=max_features,
        class_weight="balanced", random_state=SEED, n_jobs=-1
    )

def score_subset(predictions, mask):
    return f1_score(y[mask], predictions[mask], labels=CLASSES,
                    average="macro", zero_division=0) if mask.any() else np.nan

def evaluate(name, estimator, columns):
    X = data[columns]
    scores = cross_val_score(estimator, X, y, cv=repeated_splits,
                             scoring="f1_macro", error_score="raise")
    predicted = cross_val_predict(estimator, X, y, cv=fixed_splits, method="predict")
    per_class = f1_score(y, predicted, labels=CLASSES, average=None, zero_division=0)
    row = {
        "experiment": name, "features": len(columns),
        "mean_macro_f1": scores.mean(), "std_macro_f1": scores.std(),
        "oof_macro_f1": f1_score(y, predicted, labels=CLASSES, average="macro", zero_division=0),
        "normal_f1": per_class[0], "side_i_f1": per_class[1], "side_ii_f1": per_class[2],
        "high_speed_f1": score_subset(predicted, data["speed__kmh"].ge(40).to_numpy()),
        "overlap_40_50_f1": score_subset(predicted, data["speed__kmh"].between(40, 50).to_numpy()),
    }
    return row, predicted


## 6. Train the baseline models

These five comparisons answer the essential questions. The full earlier experiment history remains in the original notebooks; repeating every rejected variant is optional.

| Experiment | What it tells us |
|---|---|
| Always Normal | Minimum comparison score |
| Speed only | How much can the speed shortcut explain? |
| Vibration time + Logistic | Can a simple model use vibration measurements? |
| Vibration time + Extra Trees | Do nonlinear relationships help? |
| Vibration and shock + Logistic | Does the broader signal set support a competitive alternative? |

The models give more weight to the rare fault classes. The data is not duplicated or discarded.


In [ ]:
experiments = {
    "Always Normal": (DummyClassifier(strategy="most_frequent"), ["speed__kmh"]),
    "Speed only": (logistic(), ["speed__kmh"]),
    "Vibration time - Logistic": (logistic(), vibration_time),
    "Vibration time - Extra Trees": (trees(), vibration_time),
    "Vibration + shock - Logistic": (logistic(), all_signals),
}
results = []
oof_predictions = {}
for name, (estimator, columns) in experiments.items():
    print("Evaluating:", name, flush=True)
    row, predicted = evaluate(name, estimator, columns)
    results.append(row)
    oof_predictions[name] = predicted
baseline_results = pd.DataFrame(results)
display(baseline_results.round(3))


## 7. Look at the baseline's mistakes

Inspect the compact Extra Trees baseline. In the earlier run, its 20 errors included 9 Normal false alarms, 6 missed/mislocalised Side I files, and 5 Side II mistakes.

Compare speeds and side contrast. A fault with nearly equal vibration on both sides is harder to localise; a healthy recording at high speed can have large vibration on both sides.


In [ ]:
baseline_name = "Vibration time - Extra Trees"
error_view = data[[
    "file_id", "label", "speed__kmh", "vibration__side_log_ratio__rms_max"
]].copy()
error_view["prediction"] = oof_predictions[baseline_name]
error_view["correct"] = error_view["label"] == error_view["prediction"]
display(error_view.loc[~error_view["correct"]].sort_values(["label", "speed__kmh"]).round(3))
display(error_view.groupby(["label", "correct"]).agg(
    files=("file_id", "size"),
    median_speed=("speed__kmh", "median"),
    median_side_ratio=("vibration__side_log_ratio__rms_max", "median"),
).round(3))


## 8. Reproduce the useful refinement

The earlier investigation tested ten refinements. The strongest kept the same 15 vibration-time measurements but let each tree split consider more of them: `max_features=0.75` rather than `"sqrt"`.

This changes how the model combines the available measurements. It does not add data or introduce new features. We reproduce that comparison using the same validation splits.


In [ ]:
refined_name = "Vibration time - Refined Extra Trees"
experiments[refined_name] = (trees(max_features=0.75), vibration_time)
row, predicted = evaluate(refined_name, *experiments[refined_name])
results.append(row)
oof_predictions[refined_name] = predicted
comparison = pd.DataFrame(results).sort_values("mean_macro_f1", ascending=False)
display(comparison.round(3))

plot_data = comparison.sort_values("mean_macro_f1")
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(plot_data["experiment"], plot_data["mean_macro_f1"],
        xerr=plot_data["std_macro_f1"], color="#4C78A8")
ax.set(xlim=(0, 1), xlabel="Macro F1 (mean and fold standard deviation)",
       title="Same data and validation rules for every model")
plt.tight_layout()
plt.show()


## 9. Select the model and document the limits

The leading choice from the earlier investigation is **Extra Trees using 15 vibration-time features**, with 250 trees, minimum leaf size 2, balanced class weights, and `max_features=0.75`.

Previously this reproduced approximately:
- Repeated Macro F1: **0.813 ± 0.122**, versus **0.766 ± 0.090** for the original tree settings.
- Fixed-fold Side I F1: **0.714**, versus **0.552**.
- Fixed-fold errors: **12**, versus **20**.

Use the computed table above to check the current run. Small differences or a new leading candidate deserve review, not automatic selection.

These are results from model development, not an independent final-test score. Repeating folds with another seed uses the same recordings; it is a stability check, not new evidence from unseen data. Also, a model without an explicit speed column can still infer speed from vibration. The high-speed subset checks are encouraging, but do not eliminate that issue.


In [ ]:
selected_name = refined_name
selected_columns = list(vibration_time)
selected_estimator = trees(max_features=0.75)  # Unfitted template for the later final training step.
selected_result = comparison.set_index("experiment").loc[selected_name]
print("Selected candidate:", selected_name)
print("Measurements used:", len(selected_columns))
display(selected_result.round(3))
if comparison.iloc[0]["experiment"] != selected_name:
    print("Review needed: another experiment led the current run.")

predicted = oof_predictions[selected_name]
matrix = confusion_matrix(y, predicted, labels=CLASSES)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set(xlabel="Predicted", ylabel="Actual", title="Selected model: validation predictions")
plt.tight_layout()
plt.show()

mistakes = data[["file_id", "label", "speed__kmh"]].copy()
mistakes["prediction"] = predicted
mistakes = mistakes[mistakes["label"] != mistakes["prediction"]]
print(f"Fixed-fold mistakes: {len(mistakes)} / {len(data)}")
display(mistakes.sort_values(["label", "speed__kmh"]).round(3))


## Where this leaves the project

You now have one notebook that explains the data, trains comparison models, and selects the leading recipe.

**This notebook ends at selection.** The models fitted above are validation models. No final model artifact or submission CSV has been created by this notebook.

The next deployment step is to fit `selected_estimator` once on all 272 labelled files using `selected_columns`, then save the fitted model together with its feature order and preprocessing contract. The web app can later call a small prediction function that uses this same recipe. The app and demo video remain compulsory hackathon deliverables.

For day-to-day development, this notebook is the only entry point you need. The earlier helper files and notebooks are retained as the experiment record; this notebook runs independently of them.
